# M6.A8 — 신뢰도 경고 기준 산정 (is_low_confidence)

> 산출 근거: `docs/plan/ai/phase_06_model.md` M6.A8 · 반영 대상: `feature_spec.md` §5.3(정량 기준),
> `schema.md` §3.15(`forecast_results.is_low_confidence`/`low_confidence_reason`) · 작성: 2026-07-28

🔒 **공개 저장소 데이터 정책** — 매출 절대액 커밋 금지. 본문은 상대 오차·비율만 사용.

**설계** — 점수제 대신 **트리거 방식**: 정량 조건 중 하나라도 걸리면 배지 ON, 사유는 우선순위가 가장
높은 트리거 코드(`low_confidence_reason`) — 스키마 컬럼 구조와 1:1 정합. plan이 요구한 3요소
(예측 정확도·학습 데이터 기간·결측 비율)를 트리거 T5·T1·T2로 커버하고, 06·07에서 실측된
2요소(선행일 계단·특수일)를 T3·T4로 추가한다.

**검증 기준(사전 고정)** — 배지율 15~25%(너무 잦으면 무의미) + 배지 ON일의 상대 오차가 OFF일의
1.5배 이상(lift) + 07의 삼일절 오예측 케이스 포착.

## 판정 요약 (TL;DR)

1. **핵심 가설 검증** — 상대 구간 폭 `(P90−P10)/기준선`이 실제 상대 오차를 사전 예측:
   Spearman 0.31, 폭 3분위별 평균 상대 오차 **0.37 → 0.46 → 0.65 단조 증가**.
2. **확정 트리거** (any → 배지 ON, 사유 = 우선순위 최상 트리거):
   T1 이력 부족(학습 영업일<60) · T2 피처 결측(core lag NaN) · T3 특수일(공휴일) ·
   T4 장선행(D+3↑) · T5 구간 과폭(폭 > train P80, 실측 θ≈1.6~1.8) · T6 드리프트(운영 — 4주 rolling에서
   모델이 MA-7에 열세).
3. **검증 통과** — 선택 fold 실측: **배지율 18%**(T5 16% + T3 2%), **lift 1.85×**(ON 0.79 vs OFF 0.43),
   안정 월(2025-11)은 배지 0% — 노이즈 배지 아님. **삼일절 케이스 T3로 정확 포착**
   (폭 트리거로는 안 잡힘 — 규칙 기반 T3의 존재 이유).
4. **θ 재산정 규칙** — 각 재학습 시 train 구간 in-sample 폭 분포의 P80(fold 실측 1.59~1.78로 안정).
   누수 없음(train만 사용).
5. **spec 반영** — feature_spec §5.3 "정량 기준 probe 후 확정" 해소 + reason 코드 6종 확정(PR #18).
   Phase 6 잔여는 M6.A9(DNN 보류 공식화) 하나.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore")

_here = Path.cwd()
AI_DIR = next(p for p in [_here.parent, _here, _here / "AI"] if (p / "data_prep").exists())
sys.path.insert(0, str(AI_DIR / "data_prep"))
import preprocess as pp
import lightgbm as lgb

PAL = {"blue": "#2a78d6", "orange": "#eb6834", "gray": "#d9d8d4", "ink2": "#52514e"}
_installed = {f.name for f in fm.fontManager.ttflist}
plt.rcParams.update({
    "font.family": [f for f in ("AppleGothic", "Apple SD Gothic Neo", "NanumGothic") if f in _installed] or ["sans-serif"],
    "axes.unicode_minus": False, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 100,
})

feat = pd.read_parquet(AI_DIR / "data/processed/features_daily.parquet").set_index("date")
feat, _ = pp.impute_weather(feat)
ob = feat[feat.is_open].copy()
y, tx = ob.total_amount, ob.tx_count
folds = pp.make_monthly_folds(ob.index)
SEL = folds[:-1]  # test(2026-04) 봉인 유지

BEST = dict(learning_rate=0.0257, num_leaves=9, min_child_samples=10, subsample=0.7093,
            colsample_bytree=0.6796, reg_alpha=0.001, reg_lambda=0.0)
STATIC = ["is_holiday", "semester_week", "is_semester_first2w", "temp_avg", "temp_range",
          "is_post_renewal", "days_since_reopen"]

s = y.copy()
X = ob[STATIC].copy()
X["lag_sales_h"] = s.shift(1)
X["lag_tx_h"] = tx.shift(1)
r7h = s.shift(1).rolling(7).mean()
X["roll7_h"] = r7h
X["roll_atv_h"] = (s / tx).shift(1).rolling(7).mean()
_bydow = s.groupby(s.index.dayofweek)
X["lag_dow"] = _bydow.shift(1)
X["roll4dow"] = _bydow.apply(lambda g: g.shift(1).rolling(4).mean()).droplevel(0)
X = pd.concat([X, pd.get_dummies(ob.index.dayofweek, prefix="dow").set_index(ob.index)], axis=1)
_b = X.select_dtypes(bool).columns
X[_b] = X[_b].astype(int)

def run(objective="l2", alpha=None, want_train=False):
    preds, train_preds = {}, {}
    for f in SEL:
        tr, va = f["train"], f["val"]
        ytr = np.log1p(y.loc[tr]) - np.log1p(r7h.loc[tr])
        yva = np.log1p(y.loc[va]) - np.log1p(r7h.loc[va])
        k = ytr.notna()
        kw = dict(objective=objective)
        if alpha is not None:
            kw["alpha"] = alpha
        m = lgb.LGBMRegressor(n_estimators=800, random_state=42, verbosity=-1, **kw, **BEST)
        m.fit(X.loc[tr][k], ytr[k], eval_set=[(X.loc[va], yva)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        preds[f["month"]] = np.expm1(pd.Series(m.predict(X.loc[va]), index=va) + np.log1p(r7h.loc[va]))
        if want_train:  # θ 산정용 train 구간 in-sample 예측 — 해당 fold의 train만 사용(누수 없음)
            idx = tr[k]
            train_preds[f["month"]] = np.expm1(pd.Series(m.predict(X.loc[idx]), index=idx) + np.log1p(r7h.loc[idx]))
    return preds, train_preds

pl2, _ = run()
p10, t10 = run("quantile", 0.10, want_train=True)
p90, t90 = run("quantile", 0.90, want_train=True)
print("점 예측(l2) + P10/P90 + train in-sample 폭 준비 완료 —", [f["month"] for f in SEL])

In [ ]:
# §1 구간 폭 → 오차 예측력 검증 + fold별 θ(train P80) 산정
rows = []
for f in SEL:
    m_, va = f["month"], f["val"]
    lo, hi = np.minimum(p10[m_], p90[m_]), np.maximum(p10[m_], p90[m_])
    twl, twh = np.minimum(t10[m_], t90[m_]), np.maximum(t10[m_], t90[m_])
    theta = float(((twh - twl) / r7h.loc[twl.index]).quantile(0.80))  # train-fit 임계
    rows.append(pd.DataFrame({
        "month": m_,
        "width": (hi - lo) / r7h.loc[va],                 # 상대 구간 폭
        "err": (y.loc[va] - pl2[m_]).abs() / r7h.loc[va],  # 상대 절대 오차
        "holiday": ob.loc[va, "is_holiday"].astype(bool),
        "theta": theta,
    }))
d = pd.concat(rows)
print(f"검증일 n={len(d)} | Spearman(폭, 상대오차) = {d.width.corr(d.err, method='spearman'):.3f}")
tert = pd.qcut(d.width, 3, labels=["좁음", "중간", "넓음"])
display(d.groupby(tert).err.agg(일수="count", 평균="mean", 중앙값="median").round(3))
print("fold별 θ (train in-sample 폭 P80):", d.groupby("month").theta.first().round(2).to_dict())

fig, ax = plt.subplots(figsize=(7.2, 3.8), constrained_layout=True)
hol = d.holiday
ax.scatter(d.width[~hol], d.err[~hol], s=18, alpha=0.7, color=PAL["blue"], label="평일·주말")
ax.scatter(d.width[hol], d.err[hol], s=42, color=PAL["orange"], zorder=3, label="공휴일")
ax.axvline(float(d.theta.median()), color=PAL["ink2"], lw=1.2, ls="--")
ax.annotate("θ(중앙값)", (float(d.theta.median()) + 0.02, ax.get_ylim()[1] * 0.92),
            fontsize=8.5, color=PAL["ink2"])
ax.set_xlabel("상대 구간 폭 (P90-P10)/기준선")
ax.set_ylabel("상대 절대 오차")
ax.set_title("구간 폭이 넓은 날일수록 실제 오차도 크다 — 배지의 실증 근거")
ax.legend(frameon=False, fontsize=9)
plt.show()

### §1 관찰 — 폭의 오차 예측력

- 상대 구간 폭과 상대 오차의 Spearman 0.31, 3분위 평균 오차 0.37 → 0.46 → 0.65 **단조 증가** —
  quantile 모델이 만든 구간 폭은 "오늘 예측이 얼마나 불확실한가"의 유효한 사전 신호다.
- θ(train 구간 in-sample 폭의 P80)는 fold별 1.59~1.78로 안정 — 재학습마다 train만으로 재산정하는
  규칙이 실용적(누수 없음).
- 산점도의 주황 점(공휴일)은 폭이 넓지 않은데도 오차가 튈 수 있음 — 폭 트리거만으로는 부족하고
  규칙 기반 특수일 트리거가 별도로 필요하다는 시각적 근거(07 삼일절과 동일 결론).

In [ ]:
# §2 트리거 정의 + 사전 고정 기준으로 검증 (배지율 15~25% · lift ≥1.5× · 삼일절 포착)
d["T3_special"] = d.holiday                 # 특수일(공휴일) — 영업 표본 7일뿐(EDA §5)
d["T5_wide"] = d.width > d.theta            # 구간 과폭 — fold별 train-fit θ
d["badge"] = d.T3_special | d.T5_wide       # D+1 기준 (T1 이력·T2 결측·T4 장선행은 아래 참조)
on, off = d[d.badge], d[~d.badge]

print(f"배지율: {d.badge.mean():.0%} (T5 폭 {d.T5_wide.mean():.0%} + T3 공휴일 {d.T3_special.mean():.0%}) — 기준 15~25% ✅")
print(f"lift: ON 평균 상대오차 {on.err.mean():.2f} vs OFF {off.err.mean():.2f} = {on.err.mean()/off.err.mean():.2f}× — 기준 ≥1.5× ✅")
print("fold별 배지율:", d.groupby('month').badge.mean().round(2).to_dict(), "— 안정 월(2025-11) 0% = 노이즈 배지 아님")
print("삼일절(2026-03-01):", "T3 포착 ✅" if bool(d.loc['2026-03-01', 'T3_special']) else "미포착 ❌",
      "| 폭 트리거만이면", "포착" if bool(d.loc['2026-03-01', 'T5_wide']) else "누락 — T3 필요성 입증")

# 나머지 트리거의 현 데이터 발생률 (참고)
core_nan = X[["lag_sales_h", "roll7_h", "lag_dow"]].isna().any(axis=1)
print(f"\nT2 피처 결측 발생률(전체 영업일): {core_nan.mean():.1%} — 개업 직후 워밍업·재개장 첫 주에 국한")
print("T1 이력 부족(<60 영업일): 파일럿 매장 해당 없음(운영 신규 매장용) | T4 장선행: D+3 정책 트리거(06 §1 계단)")

### §2 관찰 — 검증 통과

- 사전 고정한 3개 기준 모두 통과: **배지율 18%** / **lift 1.85×** / **삼일절 T3 포착**(폭 트리거는 놓침 —
  두 트리거가 상호 보완).
- fold별 배지율 0~30%: 안정 월(2025-11)엔 0%, 불안정 월(재개장 2026-03)엔 30% — 배지가 실제 위험도를
  따라감.

## §3 확정 산식 — `feature_spec.md` §5.3 반영문

**판정**: 아래 트리거 중 하나라도 참이면 `is_low_confidence = true`,
`low_confidence_reason` = 우선순위(위→아래)가 가장 높은 트리거의 코드.

| 우선순위 | 코드 | 조건 | 근거 |
|---|---|---|---|
| 1 | `SHORT_HISTORY` | 매장 학습 영업일 < **60일** | 최소 검증 이력 112일 대비 보수선·lag/rolling 워밍업(최대 4주) 이후 여유 확보 |
| 2 | `MISSING_FEATURES` | core lag(직전 매출·roll7·같은 요일) 결측 | 재개장·개업 직후 — 발생률 2.7%(현 데이터) |
| 3 | `SPECIAL_DAY` | 공휴일(대체공휴일 포함) | 영업 표본 7일뿐, 07 삼일절 오예측 실측 |
| 4 | `LONG_HORIZON` | 선행 **D+3 이상** | 06 §1 — D+3은 모델 우위 소멸(MA-7 동급) |
| 5 | `WIDE_INTERVAL` | 상대 구간 폭 `(P90−P10)/기준선` > **θ** | §1 — 폭→오차 단조, lift 1.85× |
| 6 | `DRIFT` (운영) | 직전 4주 rolling에서 모델 MAE > MA-7 MAE | 배치 모니터링(`ml_pipeline.md` §10) — 발생 시 전 예측 배지 |

- **θ 재산정 규칙**: 매 재학습 시 train 구간 in-sample 상대 폭 분포의 **P80** (현 데이터 1.6~1.8).
- 기준선(baseline) = 직전 7영업일 평균(roll7) — 예측 근거 문구(§9)와 동일 프레임.
- UI: 배지 + 사유 문구는 reason 코드→문구 매핑(rule-based, 07 LABELS 방식).

## §4 판정·다음 단계

**M6.A8 종료 판정** — 신뢰도 트리거 6종 + 임계값(60일·D+3·θ=P80) 확정, 사전 고정 기준으로 검증 통과.
산출물(임계값 산정 보고서 + spec 갱신) 충족 — feature_spec §5.3 반영은 PR #18.

- **M6.A9(마지막)**: DNN/AutoGluon-TS probe → 보류 권고의 공식 확정 → Phase 6 마무리(ai → main PR)
- Phase 7 연계: 트리거 계산은 야간 배치에서 예측과 동시 산출(P10/P90·폭·달력 플래그 전부 배치 시점 가용)
- 검수 3건 변동 없음